# Systems Genetics 2026 – Final Project

Pipeline: QTL analysis (Section 2) → eQTL analysis (Section 3) → QTL/eQTL overlap (Section 4) → causality / mediation tests (Section 5).

All theoretical answers and discussion are in the report; this notebook contains code and results only.

## Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import f as f_dist, linregress

GENOTYPES_FILE = 'genotypes.xls'
PHENOTYPES_FILE = 'phenotypes.xls'
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Chosen phenotypes (Section 1).
# 881  = dorsal striatum volume adjusted for shrinkage, age, sex, plane of section and BXD epoch (Rosen et al. 2009);
# 1231 = morphine-induced locomotion, 0-180 min total, females (Philip et al. 2010).
PHENOTYPE_IDS = [881, 1231]

# Genotype encoding for the additive linear-regression model (heterozygotes considered)
GENOTYPE_ENCODING = {'B': 0.0, 'H': 1.0, 'D': 2.0}   # 'U' (unknown) -> NaN
FDR_ALPHA = 0.05          # BH-FDR significance level on q-values
MIN_STRAINS_PER_SNP = 8   # skip SNPs with fewer informative strains
N_FALLBACK_SNPS = 15      # if no significant SNP: report the 15 lowest P-values

## Section 2 – QTL analysis

The pipeline is the HW2 pipeline reduced to its essentials:

1. parse the genotype / phenotype Excel files;
2. for a phenotype ID, regress the strain means on each SNP (encoded B=0, H=1, D=2) and compute the F-test P-value of the regression;
3. correct for multiple testing with Benjamini–Hochberg FDR (q-value per SNP);
4. write a CSV with the significant SNPs (locus name + adjusted P-value), or the 15 lowest-P SNPs if none is significant.

The per-SNP F-test is vectorised over all SNPs with NumPy (sufficient statistics of the simple linear regression), which is also what Section 3 needs when the same test is run for thousands of genes.

### Parsing

In [2]:
def load_genotypes(path=GENOTYPES_FILE):
    """Genotype table: one row per SNP with Locus / Chr / position + one column per BXD strain (B/D/H/U)."""
    geno = pd.read_excel(path, header=1)
    strain_cols = [c for c in geno.columns if str(c).startswith('BXD')]
    # a handful of calls are lower-case in the file -> normalise
    geno[strain_cols] = geno[strain_cols].apply(lambda s: s.str.upper())
    geno = geno.rename(columns={'Chr_Build37': 'Chr', 'Build37_position': 'Position'})
    return geno[['Locus', 'Chr', 'Position'] + strain_cols], strain_cols


def load_phenotypes(path=PHENOTYPES_FILE):
    """Phenotype table: one row per phenotype (ID_FOR_CHECK), one column per strain (strain means)."""
    return pd.read_excel(path)


def encode_genotypes(geno, strain_cols):
    """Numeric SNP x strain matrix, B=0 / H=1 / D=2, NaN for unknown calls."""
    G = geno[strain_cols].apply(lambda s: s.map(GENOTYPE_ENCODING))   # unmapped ('U') -> NaN
    return G.to_numpy(dtype=float)


def get_phenotype_vector(pheno, phenotype_id, strain_cols):
    """Strain means of one phenotype, aligned to `strain_cols` (NaN where the strain was not measured)."""
    row = pheno.loc[pheno['ID_FOR_CHECK'] == phenotype_id]
    if row.empty:
        raise KeyError(f'phenotype {phenotype_id} not found')
    row = row.iloc[0]
    y = pd.to_numeric(row.reindex(strain_cols), errors='coerce').to_numpy(dtype=float)
    return y, row['Phenotype']


genotypes, STRAINS = load_genotypes()
phenotypes = load_phenotypes()
STRAINS = [s for s in STRAINS if s in phenotypes.columns]      # strains present in both files
G_ALL = encode_genotypes(genotypes, STRAINS)

print(f'{len(genotypes)} SNPs x {len(STRAINS)} BXD strains; {len(phenotypes)} phenotypes')
print('unknown genotype calls:', f'{np.isnan(G_ALL).mean():.1%}')
genotypes.head(3)

3796 SNPs x 92 BXD strains; 2885 phenotypes
unknown genotype calls: 3.3%


,Locus,Chr,Position,BXD1,BXD2,BXD5,BXD6,BXD8,BXD9,BXD11,...,BXD94,BXD95,BXD96,BXD97,BXD98,BXD99,BXD100,BXD101,BXD102,BXD103
0,rs6269442,1,3482276,B,B,D,D,D,B,B,...,D,D,B,B,B,B,B,U,U,U
1,rs6365999,1,4811063,B,B,D,D,D,B,B,...,D,D,B,B,B,B,B,U,U,U
2,rs6376963,1,5008090,B,B,D,D,D,B,B,...,D,D,B,B,B,B,B,U,U,U


### Association test (F-test of the linear regression) and BH-FDR

In [3]:
def f_test_all_snps(G, y, min_n=MIN_STRAINS_PER_SNP):
    """F-test of the simple linear regression y ~ b0 + b1 * genotype, for every SNP at once.

    G : (n_snps, n_strains) numeric genotypes (0/1/2, NaN = unknown)
    y : (n_strains,) phenotype values (NaN = strain not measured)
    Returns (p_values, n_strains_used); NaN P-value for SNPs that are monomorphic
    in the measured strains or have fewer than `min_n` informative strains.
    F = (SS_regression / 1) / (SS_residual / (n - 2)),  F ~ F(1, n-2) under H0: b1 = 0.
    """
    measured = ~np.isnan(y)
    G, y = G[:, measured], y[measured]
    known = ~np.isnan(G)
    x = np.where(known, G, 0.0)
    yy = np.where(known, y, 0.0)              # y masked per SNP by its known genotypes
    n = known.sum(axis=1)
    Sx, Sy = x.sum(1), yy.sum(1)
    Sxx, Syy, Sxy = (x * x).sum(1), (yy * yy).sum(1), (x * yy).sum(1)
    with np.errstate(invalid='ignore', divide='ignore'):
        Sxx_c = Sxx - Sx ** 2 / n            # centred sums of squares / cross-products
        Syy_c = Syy - Sy ** 2 / n
        Sxy_c = Sxy - Sx * Sy / n
        ss_reg = Sxy_c ** 2 / Sxx_c
        ss_res = Syy_c - ss_reg
        F = ss_reg / (ss_res / (n - 2))
        p = f_dist.sf(F, 1, n - 2)
    p[(n < min_n) | ~(Sxx_c > 0)] = np.nan
    return p, n


def bh_fdr(p):
    """Benjamini-Hochberg adjusted P-values (q-values). NaNs are ignored and returned as NaN."""
    p = np.asarray(p, dtype=float)
    q = np.full_like(p, np.nan)
    valid = ~np.isnan(p)
    pv = p[valid]
    m = pv.size
    order = np.argsort(pv)
    adj = pv[order] * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]   # enforce monotonicity
    out = np.empty(m)
    out[order] = np.minimum(adj, 1.0)
    q[valid] = out
    return q


# sanity check: the vectorised F-test equals scipy's per-SNP regression P-value
y_check, _ = get_phenotype_vector(phenotypes, PHENOTYPE_IDS[0], STRAINS)
p_check, _ = f_test_all_snps(G_ALL, y_check)
for i in [0, 500, 1500, 3000]:
    ok = ~np.isnan(G_ALL[i]) & ~np.isnan(y_check)
    assert np.isclose(p_check[i], linregress(G_ALL[i][ok], y_check[ok]).pvalue)
print('vectorised F-test matches scipy.stats.linregress')

vectorised F-test matches scipy.stats.linregress


### QTL pipeline

In [4]:
def run_qtl_pipeline(phenotype_id, alpha=FDR_ALPHA, save=True, verbose=True):
    """Full QTL scan for one phenotype. Returns a DataFrame (one row per SNP) sorted by P-value,
    and writes the significant SNPs (or the 15 lowest-P SNPs) to results/qtl_<id>.csv."""
    y, name = get_phenotype_vector(phenotypes, phenotype_id, STRAINS)
    p, n = f_test_all_snps(G_ALL, y)
    res = genotypes[['Locus', 'Chr', 'Position']].copy()
    res['n_strains'] = n
    res['p_value'] = p
    res['q_value'] = bh_fdr(p)
    res['significant'] = res['q_value'] < alpha
    res = res.sort_values('p_value').reset_index(drop=True)

    n_sig = int(res['significant'].sum())
    out = res[res['significant']] if n_sig > 0 else res.head(N_FALLBACK_SNPS)
    if save:
        out[['Locus', 'q_value']].rename(columns={'q_value': 'adjusted_p_value'}).to_csv(
            f'{RESULTS_DIR}/qtl_{phenotype_id}.csv', index=False)
    if verbose:
        print(f'Phenotype {phenotype_id}: {name.strip()}')
        print(f'  strains measured: {int((~np.isnan(y)).sum())}  |  SNPs tested: {int(res.p_value.notna().sum())}')
        print(f'  significant SNPs (q < {alpha}): {n_sig}' + ('' if n_sig else f'  -> reporting {N_FALLBACK_SNPS} lowest P-values'))
        if n_sig:
            print('  significant SNPs per chromosome:', res[res.significant].groupby('Chr').size().to_dict())
    return res

### QTL scan of the two chosen phenotypes

In [5]:
qtl_results = {pid: run_qtl_pipeline(pid) for pid in PHENOTYPE_IDS}

Phenotype 881: Central nervous system, morphology: Striatum volume, dorsal striatum, bilateral and adjusted for shrinkage, age, sex, plane of section, and BXD group/epoch (mm^3)
  strains measured: 53  |  SNPs tested: 3796
  significant SNPs (q < 0.05): 24
  significant SNPs per chromosome: {6: 24}
Phenotype 1231: Morphine response (50 mg/kg ip), locomotion from 0-180 min (total activity over 3 hour test) after injection in an activity chamber for females [cm]
  strains measured: 64  |  SNPs tested: 3796
  significant SNPs (q < 0.05): 38
  significant SNPs per chromosome: {5: 7, 10: 25, 11: 6}


In [6]:
# Significant SNPs (or 15 lowest P) per phenotype - the content of results/qtl_<id>.csv
for pid, res in qtl_results.items():
    out = res[res.significant] if res.significant.any() else res.head(N_FALLBACK_SNPS)
    print(f'\n=== Phenotype {pid}: {len(out)} SNPs written to {RESULTS_DIR}/qtl_{pid}.csv ===')
    display(out[['Locus', 'Chr', 'Position', 'n_strains', 'p_value', 'q_value']].reset_index(drop=True))


=== Phenotype 881: 24 SNPs written to results/qtl_881.csv ===


,Locus,Chr,Position,n_strains,p_value,q_value
0,rs13478889,6,91386029,53,0.000015,0.012110
1,rs3661039,6,90308024,53,0.000015,0.012110
2,rs13478880,6,88537081,53,0.000018,0.012110
3,rs3713705,6,88668211,53,0.000018,0.012110
4,rs4226061,6,86579760,53,0.000019,0.012110
5,rs13475374,6,86953033,53,0.000019,0.012110
6,rs13478871,6,85767550,53,0.000028,0.013520
7,CEL-6_86437630,6,86058923,53,0.000028,0.013520
8,gnf06.086.089,6,87099761,53,0.000036,0.013841
9,rs13478876,6,87384144,53,0.000036,0.013841



=== Phenotype 1231: 38 SNPs written to results/qtl_1231.csv ===


,Locus,Chr,Position,n_strains,p_value,q_value
0,rs13480484,10,8189404,64,0.000002,0.001804
1,rs4228079,10,4609418,64,0.000002,0.001804
2,rs3721803,10,3342586,64,0.000002,0.001804
3,rs3695003,10,7632997,64,0.000002,0.001804
4,D10Mit28,10,9133173,64,0.000003,0.002063
5,rs6185923,10,5378623,64,0.000003,0.002063
6,rs13481187,11,100513550,64,0.000023,0.011003
7,rs13481186,11,100224674,64,0.000023,0.011003
8,rs3664101,10,9329071,64,0.000094,0.025832
9,rs4139678,10,12004986,64,0.000094,0.025832


## Section 3 – eQTL analysis
*(to do)*

## Section 4 – Combining QTL and eQTL results
*(to do)*

## Section 5 – Causality analysis
*(to do)*